<a href="https://colab.research.google.com/github/rastri-dey/Ground-up-implementations-ML-algorithms-/blob/main/notebooks/RNN_scratch_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Description

Building a language model for text generation using RNN from scratch

**ML Algorithm**: RNN from scratch <br>
**Dataset**: Book by H G Wells "The Time Machine" <br>
**Framework**: Pure Python implementation of RNN using PyTorch (only for computation efficiency), without any built-in module

## Import Libraries

In [17]:
import random
import re
import collections
import os
import requests
import hashlib
import torch
import torch.nn as nn
from torch.nn import functional as F

# Data

In [18]:
# Download method is taken from another notebook
# If there is an existing already downloaded text, no need to learn this

def download(name, cache_dir=os.path.join("..", "data")):
    """Download a file inserted into DATA_HUB, return the local filename."""
    assert name in DATA_HUB, f"{name} does not exist in {DATA_HUB}."
    url, sha1_hash = DATA_HUB[name]
    os.makedirs(cache_dir, exist_ok=True)
    fname = os.path.join(cache_dir, url.split("/")[-1])
    if os.path.exists(fname):
        sha1 = hashlib.sha1()
        with open(fname, "rb") as f:
            while True:
                data = f.read(1048576)
                if not data:
                    break
                sha1.update(data)
        if sha1.hexdigest() == sha1_hash:
            return fname  # Hit cache
    print(f"Downloading {fname} from {url}...")
    r = requests.get(url, stream=True, verify=True)
    with open(fname, "wb") as f:
        f.write(r.content)
    return fname

## Terminologies

**tokens**: Each time step corresponds to 1 token. In a character level tokenization, each character is a token. <br>
**corpus**: corpus is a single list of token indices from the entire book, represented as numerical indices based on the vocabulary <br>
**vocab**: vocab is the vocabulary of The Time Machine corpus. In character level language modelling, it is a set of all characters in the book. (There is maximum of 256 ASCII characters). So this is bounded by 256 maximum.<br>

In [19]:
def read_text():
  '''
  Inputs: Text Book - Book by H G Wells "The Time Machine"
  Outputs: List of strings where each string is Cleaned-up lowercase corresponding to a line from the input file
  Process: Remove any character that is not (^) A-Z and a-z, .strip() removes leading trailing whitespaces, newline charcter and make all english characters lower
  '''
  with open(download("time_machine"), "r") as f:
    lines = f.readlines()
  return [re.sub("[^A-Za-z]+"," ", line).strip().lower() for line in lines]

def tokenize(lines, token="word"):
  '''
  Inputs: List of strings (where each string is one line from the book)
  Outputs: List of List of all tokens like [['the', 'time', 'machine'], ['by', 'h', 'g', 'wells']]- words or characters (All the words or chars used in the book)
  '''
  if (token == "word"):
    return [line.split() for line in lines]
  elif (token == "char"):
    return [list(line) for line in lines]
  else:
    print("Error: Unknown token type: " + token)

def count_corpus(tokens):
  '''
  Inputs: A 2D list of tokens
  Outputs: A dictionary object of all tokens and their frequencies from the entire book
  Process: Flatten 1D list of all tokens -> Count the frequency of each token through the collections.Counter object which in itself is a dictionary
  '''
  if len(tokens)==0 or isinstance(tokens[0], list):
    tokens_flat = [token for line in tokens for token in line] # Single List of all tokens
  return collections.Counter(tokens_flat)

In [20]:
class Vocab:
  '''
  Create a Vocab class, which assigns a index to each token: It is a dictionary of token to index and index to token
  When vocab is called with a token like Vocab(token) it would return the index of that token
  By Design the Vocab dictionary of token and index is in max to min frequency order, so by index highest frequency tokens appears before the lower frequency tokens
  '''
  def __init__(self, tokens=None, min_freq=0, reserved_tokens=None):
    if tokens == None:
      tokens = []
    if reserved_tokens == None:
      reserved_tokens = []
    counter = count_corpus(tokens)
    freq_tokens = sorted(counter.items(), key = lambda freq: freq[1], reverse=True)
    self.unk_ind, unk_tokens = 0, ["<unk>"] + reserved_tokens
    unk_tokens += [token for token, freq in freq_tokens if freq>=min_freq and token not in unk_tokens]
    self.token_idx, self.idx_token = dict(), []
    for token in unk_tokens:
      self.idx_token.append(token)
      self.token_idx[token] = len(self.idx_token) - 1

  def __len__(self):
      return len(self.idx_token)

  def __getitem__(self, tokens):
    '''
    Returns a list of token indices corresponding to the token list (If the token list is 2D, this list is 2D)
    '''
    if not isinstance(tokens, (list, tuple)):
        return self.token_idx.get(tokens, self.unk_ind)   # If there is no index for that token return 0
    return [self.__getitem__(token) for token in tokens]

  def to_tokens(self, ids):
    '''
    Returns a list of token corresponding to the token indices (If the token indices list is 2D, this list is 2D)
    '''
    if not isinstance(ids, (list, tuple)):
        return self.idx_token[ids]
    return [self.idx_token[id] for id in ids]


In [21]:
def load_corpus_data(max_tokens=-1):
    '''
    Return a single list of token indices corresponding to the book and create a vocabulary from the book
    '''
    lines = read_text()              # List of strings (where each string is one line from the book)
    tokens = tokenize(lines, "char") # 2D List of tokens (Inner 1D list of tokens is each line from the book)
    vocab = Vocab(tokens)            # Create an instance (or object) of Vocab class

    corpus = [vocab[token] for line in tokens for token in line]

    if max_tokens > 0:
        corpus = corpus[:max_tokens]

    return corpus, vocab

## Data Batch Processing

### Random Sampling of sequences within mini batches
**corpus**: Entire list of characters <br>
num_subseqs: partitioning the entire list into small subsequences of num_steps length <br>
**num_steps**: length of one subsequence <br>
initial_indices: first index of each subsequence of the entire list of characters in the book. Like if each subsequence length is 5, then this list is : `[0, 5, 10, 15, ....]` <br>
**num_batches**: how many batches of data we need. Like we may want to send the whole corpus in two batches. <br>
**Random Sampling**: Shuffle the initial_indices list randomly like `[10,0,5,15]`. So, first batch will have `[10,0]`, 2nd batch will have `[5,15]`. This ensures not only two adjacent subsequences of two different mini-batches are not really adajacent in corpus, but also 2 adjacent subsequence within same mini-batch might not be adajacent within the corpus. <br>

**`yield`** keyword within a function makes the function a generator/iterator. So when the function is being called using a for loop, at each loop it gives the yield values, pauses its execution, saves the local states until the next iteration reaches to yield

```
def simple_generator():
    yield 1
    yield 2
    yield 3

# Using the standalone generator
for value in simple_generator():
    print(value)
```

SeqDataLoader class itself is iterable because its `__iter__` method returns an iterator/generator. The seq_data_iter_random is a generator function because it uses yield, which *generates the batches of data (X,Y) in every iteration of the for loop*.

In [22]:
def seq_random_sampl(corpus, num_steps, batch_size):
  '''
  Inputs: The textbook data
  Outputs: Batch data: Input data X(sequence of characters) and corresponding labels Y(expected next sequence of characters, given last input)
  Process: Random sampling of sequences of data
  '''
  corpus = corpus[random.randint(0,num_steps-1):] # Based on the Book details, we need the corpus to start from different random starting points
  num_seqs = (len(corpus)-1)//num_steps
  initial_indices = list(range(0, num_seqs*num_steps, num_steps))
  random.shuffle(initial_indices)

  num_batches = num_seqs//batch_size

  def data(pos):
    return corpus[pos:pos+num_steps]

  for i in range(num_batches):
    rand_batch_indices = initial_indices[i:i+batch_size]
    X = [data(pos) for pos in rand_batch_indices]
    Y = [data(pos+1) for pos in rand_batch_indices]
    yield torch.tensor(X), torch.tensor(Y)

### Sequential Sampling of sequences within mini batches

In [23]:
def seq_sequential_sampl(corpus, num_steps, batch_size):
  '''
  Inputs: The textbook data
  Outputs: Batch data: Input data X(sequence of characters) and corresponding labels Y(expected next sequence of characters, given last input)
  Process: Sequential sampling of sequences of data
  '''
  offset = random.randint(0, num_steps-1)

  num_tokens = ((len(corpus)-offset-1)//batch_size)*batch_size  # Intention is to make the num_tokens a multiple of batch size, so that matrix is even, -1 is done to consider for the final label char of final input char

  Xs = torch.tensor(corpus[offset:num_tokens])     # A list
  Ys = torch.tensor(corpus[offset+1:num_tokens+1]) # A list
  Xs = Xs.reshape(batch_size, -1)                  # Matrix would be even, because of multiple of num_tokens calculation
  Ys = Ys.reshape(batch_size, -1)

  num_batches = Xs.shape[1]//num_steps

  for i in range(num_batches):
    X = Xs[:, i : i+num_steps]
    Y = Ys[:, i : i+num_steps] # No need of doing pos+1, since its already taken in tensor list Ys
    yield X, Y

In [24]:
class DataLoader:
    '''Create your own DataLoader for loading batches of (Inputs, Labels) iteratively'''

    def __init__(self, batch_size, num_steps, random_sampl, max_tokens):
        if random_sampl:
            self.data_iter_fn = seq_random_sampl
        else:
            self.data_iter_fn = seq_sequential_sampl
        self.corpus, self.vocab = load_corpus_data(max_tokens)
        self.batch_size, self.num_steps = batch_size, num_steps

    def __iter__(self):
        return self.data_iter_fn(self.corpus, self.batch_size, self.num_steps)

In [25]:
# Not learning the textbook data download part
DATA_HUB = dict()
DATA_URL = "http://d2l-data.s3-accelerate.amazonaws.com/"
DATA_HUB["time_machine"] = (DATA_URL + "timemachine.txt", "090b5e7e70c295757f55df93cb0a180b9691891a")

batch_size, num_steps = 32, 35
'''
Process: If we iterate train_iter in a for loop, it would take the __iter__ method from the DataLoader class
and iterate the sequential functions, which in return would yield (X,Y) in batches
'''
train_iter = DataLoader(batch_size, num_steps, random_sampl=False, max_tokens=-1) # Create an instance of DataLoader class
vocab = train_iter.vocab


# Model

Model has the form: <br>
$H_t = \phi(X_tW_{xh} + H_{t-1}W_{hh}+b_h)$ <br>
$O_t = H_tW_{hq}+b_q$ <br>
where, where $X_t$ is the $(n,d)$ matrix of (one-hot) inputs
(for batch size $n$ and vocabulary size $d$), $H_t$ is the hidden state matrix of size $(n,h)$ where $h$ is the number of hidden states and $O_t$ is the output of batch_size and vocab_size at current time step $t$ with matrix of size $(n,d)$ where $d$ is the vocab size ($q$ is like the output labels, here $q=d$)

### Model Parameters (Weights & Biases)

In [26]:
'''
Returns the model paarameters: W and b are weights and biases respectively
'''
def get_params(vocab_size, num_hidden_states, device):
  W_xh = torch.randn((vocab_size, num_hidden_states), device=device) * 0.01          # [N,H]
  W_hh = torch.randn((num_hidden_states, num_hidden_states), device=device) * 0.01   # [H,H]
  b_h = torch.zeros(num_hidden_states, device=device)                                # [H]: when 1D tensor is added to a matrix, pytorch handles the shape for b_h(1*H) to be added to all N batch_size rows
  W_hq = torch.rand((num_hidden_states, vocab_size), device=device) * 0.01           # [H, D] D is the vocab size (if tokens is char, then this is bounded by 26+26 uppercase, lowercase english characters from the textbook used for training)
  b_q = torch.zeros(vocab_size, device=device)                                       # [D]: O_t is the output

  params = [W_xh, W_hh, b_h, W_hq, b_q]         # List of pytorch tensors

  for param in params:
    param.requires_grad_(True)                  # For these pytorch tensors we want to enable the gradient calculation during backward traversal of loss function computation

  return params

### Initial Hidden State
In case sequential data is not used for training or we are begining the training, then we call this init_hidden_state again and again in the training loop

In [27]:
def init_hidden_state(batch_size, num_hidden_states, device):
  return (torch.zeros((batch_size, num_hidden_states), device=device),) # Output as Tuple

### Forward Function

In [28]:
 def rnn(inputs, hidden_state, params):
  '''
  Input:
  inputs: inputs is one batch of size (T, N, V), T is the num_steps, N is the batch_size, V is the vocab_size
  hidden_state: hidden_state is of size (N, H), H is the number of hidden states
  params: Model weights, biases

  Output: Returns O_t of size (T*N, V), hidden state in a tuple format
  [At t_0 Batch_size of N rows, D cols],
  [At t_1 Batch_size of N rows, D cols],
  .
  .
  .
  [At t_T Batch_size of N rows, D cols]
  '''
  W_xh, W_hh, b_h, W_hq, b_q = params
  (state,) = hidden_state                # We want to define the state as a tuple to generalize it to different kind of RNNs like LSTM, GRU etc.
  outputs = []
  for X in inputs:
    # Here X is the input at each time step: it becomes (n,d) as required per the model equations
    state = torch.tanh(torch.mm(X, W_xh) + torch.mm(state, W_hh) + b_h)  # [N,H]
    O = torch.mm(state, W_hq) + b_q                                      # Logits: [N,D]
    outputs.append(O)                                                    # List of size T torch tensors, where each tensor is of shape (N,D)

  return torch.cat(outputs, dim=0), (state, )  # Output of shape(N*T, D)

## RNN Model Class - Scratch

In [29]:
class RNN_scratch:
  def __init__(self, init_hidden_state_fn, get_params, vocab_size, num_hidden_states, forward_fn, device):
    self.vocab_size, self.num_hidden_states = vocab_size, num_hidden_states
    self.params = get_params(vocab_size, num_hidden_states, device)
    self.init_state_fn, self.forward_fn = init_hidden_state_fn, forward_fn

  def __call__(self, X, state):
    '''
    Process:
    N, T: batch_size, num_steps
    We are reshaping the original batch processed input from (N,T) to (T,N) and then converting it to one-hot encoded vector where elements are either 1 or 0
    Why .type(torch.float32) conversion is required, if reshaping was the only requirement and the elements are 0 or 1?
    Because the torch mat multiplication is assumed to be in the format of float32, so if we don't do conversion first then integer multiplication with float_32
    for model equations will give error.
    '''
    inputs = F.one_hot(X.T, self.vocab_size).type(torch.float32)
    return self.forward_fn(inputs, state, self.params)

  def init_state(self, batch_size, device):
    '''
    Process: batch_size is not part of constructor, so when this function is called from an instance of RNN_scratch, put it in the inputs of fn
    '''
    return self.init_state_fn(batch_size, self.num_hidden_states, device)


In [30]:
if torch.cuda.is_available():
  device = torch.device('cuda:0')
else:
  device = torch.device('cpu')

vocab_size = len(train_iter.vocab)
num_hidden_states = 512
model = RNN_scratch(init_hidden_state, get_params, vocab_size, num_hidden_states, rnn, device)

# Training
The standard training process for any model involves, defining the model, iterating through the epochs, iterating through the batch of data within each epoch, get the output logits, calculate the loss wrt labels, calculate the loss backwards (and store the gradients-pytorch handles it through requires_grad), update the model parameters, zero the gradients for next batch. <br>
These steps are still standardized for RNN with the additional steps for the calculation of hidden states. We will also implement our own gradient clipping to avoid the *'exploding gradients'* problem, when we want to predict a long sequence of data. <br>

1. Backpropagation for RNN happens through time sequences of data.
2. In the Forward function, the same set of weights and biases is used for each step of the sequence.
3. When backpropagation is performed, the loss is computed for each time step. Since, in forward pass fixed weights are used, to calculate the gradient of the total loss (summed over all time steps in the batch) with respect to a shared parameter (like $W_{xh}$), the gradient is computed by applying the chain rule backwards through time from the final time step to the first.
4. For a parameter like $W_{xh}$, which is used at every time step, its gradient with respect to the total loss is the sum of the gradients of the loss at each time step with respect to $W_{xh}$:

$\delta(L)/\delta(W_{xh}) = \delta(L_1)/\delta(W_{xh}) + \delta(L_2)/\delta(W_{xh}) +...+\delta(L_T)/\delta(W_{xh}) $ <br>

*Or to summarize* <br>

$ \frac{\partial L}{\partial W_{xh}} = \sum_{t=1}^T \frac{\partial L_t}{\partial W_{xh}} $

5. Then the parameters (weights) update follows: <br>

$ W_{xh}^{(new)} = W_{xh}^{(old)} - \eta \cdot \frac{\partial L}{\partial W_{xh}} $ <br>

As the gradient is computed over a batch, here we do the parameter update like:
 <br>
$ W_{xh}^{(new)} = W_{xh}^{(old)} - \frac{\text{lr}}{\text{batch_size}} \cdot \frac{\partial L}{\partial W_{xh}} $


In [31]:
# Gradient Clipping
def grad_clip(theta, model):
  params = model.params
  norm = torch.sqrt(sum(torch.sum((p.grad**2)) for p in params)) # Normalized value over all weights & biases: W_xh, W_hh, b_h, W_hq, b_q
  if norm > theta:
    for param in params:
      param.grad[:] *= theta/norm # theta is the threshold


In [46]:
def predict(model, prefix, num_preds, device):
  '''
  Input: We need a model to predict for the prefix string upto num_preds after the prefix string in the same device where, state, parameters, inputs are
  Output: Returns the prediction upto num_preds step
  Process: This is just a prediction function, model training doesn't happen here.
  We train the model in batches, but prediction happens 1 data per 1 batch, so we initialize batch_size=1 and we predict 1 data per time step
  '''
  state = model.init_state(batch_size=1, device=device)
  outputs = [vocab[prefix[0]]]
  get_input = lambda: torch.tensor([outputs[-1]], device=device).reshape(1,1) # We expect the forward fn to have the input data in a batch like (N,T) format
  for i in prefix[1:]:
    _, state = model(get_input(),state) # We do not want to pass any argument to the lambda function, it just takes the last character of output
    outputs.append(vocab[i]) # We call the inputs as the outputs here, because we keep the same outputs as the prefix and then start the prediction after prefix ends. So the total output is prefix + new prediction uptil num_preds
  for _ in range(num_preds):
    y, state = model(get_input(),state)        # y is in shape (N*T, D) = (1*1, D)
    outputs.append(int(y.argmax(dim=1).reshape(1))) # We want the max index

  return "".join(vocab.to_tokens(outputs))

predict(model, "time traveller ", 10, device)

'time traveller aaaaaaaaaa'

In [ ]:
def train_epoch(data_loader, model, loss_fn, optimizer, use_random_sampl, device):
  '''
  We detach state (tuples of state), to avoid gradient tracking on this state tensor calculation
  During sequential sampling, we want to pass the state from one mini-batch to the next. But we don't want to track the gradient changes on this,
  as this would lead to accumulation of gradients over long sequences of data which would lead to issues like exploding or vanishing gradients
  '''
  state = None
  for X, Y in data_loader:
    if state is None or use_random_sampl:
      model.init_state(batch_size=X.shape[0], device=device)
    else:
      for s in state:
        s.detach_()




In [ ]:
# PyTorch computes the gradients of the loss with respect to each of these parameters and stores them in their respective .grad attributes

### Helper code: We don't learn this

In [ ]:
# We have taken this part from another notebook, we don't learn this
class Animator:
    """For plotting data in animation."""

    def __init__(
        self,
        xlabel=None,
        ylabel=None,
        legend=None,
        xlim=None,
        ylim=None,
        xscale="linear",
        yscale="linear",
        fmts=("-", "m--", "g-.", "r:"),
        nrows=1,
        ncols=1,
        figsize=(3.5, 2.5),
    ):
        # Incrementally plot multiple lines
        if legend is None:
            legend = []
        display.set_matplotlib_formats("svg")
        self.fig, self.axes = plt.subplots(nrows, ncols, figsize=figsize)
        if nrows * ncols == 1:
            self.axes = [
                self.axes,
            ]
        # Use a lambda function to capture arguments
        self.config_axes = lambda: set_axes(self.axes[0], xlabel, ylabel, xlim, ylim, xscale, yscale, legend)
        self.X, self.Y, self.fmts = None, None, fmts

    def add(self, x, y):
        # Add multiple data points into the figure
        if not hasattr(y, "__len__"):
            y = [y]
        n = len(y)
        if not hasattr(x, "__len__"):
            x = [x] * n
        if not self.X:
            self.X = [[] for _ in range(n)]
        if not self.Y:
            self.Y = [[] for _ in range(n)]
        for i, (a, b) in enumerate(zip(x, y)):
            if a is not None and b is not None:
                self.X[i].append(a)
                self.Y[i].append(b)
        self.axes[0].cla()
        for x, y, fmt in zip(self.X, self.Y, self.fmts):
            self.axes[0].plot(x, y, fmt)
        self.config_axes()
        display.display(self.fig)
        display.clear_output(wait=True)


class Timer:
    """Record multiple running times."""

    def __init__(self):
        self.times = []
        self.start()

    def start(self):
        """Start the timer."""
        self.tik = time.time()

    def stop(self):
        """Stop the timer and record the time in a list."""
        self.times.append(time.time() - self.tik)
        return self.times[-1]

    def avg(self):
        """Return the average time."""
        return sum(self.times) / len(self.times)

    def sum(self):
        """Return the sum of time."""
        return sum(self.times)

    def cumsum(self):
        """Return the accumulated time."""
        return np.array(self.times).cumsum().tolist()


class Accumulator:
    """For accumulating sums over `n` variables."""

    def __init__(self, n):
        self.data = [0.0] * n

    def add(self, *args):
        self.data = [a + float(b) for a, b in zip(self.data, args)]

    def reset(self):
        self.data = [0.0] * len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


def set_axes(axes, xlabel, ylabel, xlim, ylim, xscale, yscale, legend):
    """Set the axes for matplotlib."""
    axes.set_xlabel(xlabel)
    axes.set_ylabel(ylabel)
    axes.set_xscale(xscale)
    axes.set_yscale(yscale)
    axes.set_xlim(xlim)
    axes.set_ylim(ylim)
    if legend:
        axes.legend(legend)
    axes.grid()


def sgd(params, lr, batch_size):
    """Minibatch stochastic gradient descent."""
    with torch.no_grad():
        for param in params:
            param -= lr * param.grad / batch_size
            param.grad.zero_()